# 04 — NMF Metagenes

Non-negative matrix factorisation decomposes the expression matrix into:
- **W** (genes × k): metagene signatures — biologically interpretable gene programs
- **H** (k × samples): sample usage — how much each sample uses each metagene

We select k using reconstruction error + cophenetic correlation stability.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import NMF

PROJECT_ROOT = '/content/gene-expression-la-analysis'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

from linear_algebra import run_nmf, top_metagene_genes
from visualization import (plot_metagene_weights, plot_nmf_H_heatmap,
                            plot_expression_heatmap)

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')


In [ ]:
# ── Load processed data ────────────────────────────────────────────────────
expr = pd.read_csv(f'{PROJECT_ROOT}/data/processed/processed_expression.csv',
                   index_col=0)
meta = pd.read_csv(f'{PROJECT_ROOT}/data/processed/metadata_clean.csv',
                   index_col=0)

common = expr.columns.intersection(meta.index)
expr   = expr[common]
meta   = meta.loc[common]

# NMF requires non-negative data
X = expr.values.astype(float)
if X.min() < 0:
    X = X - X.min()
    expr = pd.DataFrame(X, index=expr.index, columns=expr.columns)

print(f'Expression (non-negative): {expr.shape}')


## 1. Select Optimal k (Reconstruction Error)

In [ ]:
k_range = range(2, 16)
errors  = []

for k in k_range:
    nmf = NMF(n_components=k, init='nndsvda', max_iter=300, random_state=42)
    nmf.fit(expr.values)
    errors.append({'k': k, 'reconstruction_error': nmf.reconstruction_err_})
    print(f'k={k:2d}  error={nmf.reconstruction_err_:.4f}')

err_df = pd.DataFrame(errors)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(err_df['k'], err_df['reconstruction_error'], 'o-', color='steelblue')
ax.set_xlabel('Number of Metagenes (k)')
ax.set_ylabel('Reconstruction Error')
ax.set_title('NMF — Reconstruction Error vs k')

# Mark elbow (largest drop)
diffs = np.diff(err_df['reconstruction_error'].values)
elbow = err_df['k'].values[1:][np.argmin(diffs)]
ax.axvline(elbow, ls='--', color='crimson', label=f'Elbow k={elbow}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/04_nmf_elbow.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Suggested k (elbow): {elbow}')


## 2. Run NMF with Selected k

In [ ]:
K = int(elbow)   # or set manually, e.g. K = 7
print(f'Running NMF with k = {K}')

nmf_result = run_nmf(expr, n_components=K, max_iter=500)
W = nmf_result['W']   # genes × k
H = nmf_result['H']   # k × samples


In [ ]:
# ── H matrix heatmap ──────────────────────────────────────────────────────
LABEL_COL = 'source_name_ch1'   # adjust to your metadata column
col_labels = meta.get(LABEL_COL, None)

plot_nmf_H_heatmap(H, col_labels=col_labels,
                   title=f'NMF H Matrix (k={K}) — Sample Metagene Usage',
                   save_path=f'{PROJECT_ROOT}/data/results/04_nmf_H_heatmap.png')


## 3. Inspect Each Metagene

In [ ]:
# Plot top-25 genes for every metagene
for mg in range(1, K+1):
    plot_metagene_weights(W, metagene=mg, top_n=25,
                          save_path=f'{PROJECT_ROOT}/data/results/04_metagene{mg}_weights.png')


In [ ]:
# ── Top genes per metagene (table) ─────────────────────────────────────────
top_genes_per_mg = {}
for mg in range(1, K+1):
    tg = top_metagene_genes(nmf_result, metagene=mg, top_n=20)
    top_genes_per_mg[f'Metagene{mg}'] = tg.index.tolist()

top_df = pd.DataFrame(top_genes_per_mg)
top_df.to_csv(f'{PROJECT_ROOT}/data/results/04_top_metagene_genes.csv')
print(top_df.head(10))


## 4. Sample Clustering by Dominant Metagene

In [ ]:
dominant = H.idxmax(axis=0)   # which metagene dominates each sample
print('Dominant metagene distribution:')
print(dominant.value_counts())

# PCA scatter coloured by dominant metagene
try:
    pca_scores = pd.read_csv(f'{PROJECT_ROOT}/data/processed/pca_scores.csv',
                              index_col=0)
    from visualization import plot_pca_scatter
    common2 = pca_scores.index.intersection(dominant.index)
    plot_pca_scatter(pca_scores.loc[common2], labels=dominant.loc[common2],
                     title='PCA coloured by Dominant Metagene',
                     save_path=f'{PROJECT_ROOT}/data/results/04_pca_by_metagene.png')
except FileNotFoundError:
    print('Run notebook 03 first to generate PCA scores.')


In [ ]:
# ── Save NMF results ──────────────────────────────────────────────────────
W.to_csv(f'{PROJECT_ROOT}/data/processed/nmf_W.csv')
H.to_csv(f'{PROJECT_ROOT}/data/processed/nmf_H.csv')
dominant.to_csv(f'{PROJECT_ROOT}/data/processed/nmf_dominant_metagene.csv')
print('NMF results saved.')
